[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/05_global_parameters.ipynb)

# 5 · Global atmospheric parameters

*Brown Dwarf Spectra On-Ramp · about 50 minutes*

**By the end of this notebook you will be able to:**
- say what effective temperature, gravity, metallicity, clouds, and vertical mixing each do to a spectrum
- read a parameter stack and a ratio-to-fiducial plot
- compare real low-gravity and low-metallicity objects with field objects of the same type
- recognize degeneracies: different parameter changes that look alike

> **First time here?** Do *File → Save a copy in Drive* before you change anything (notebook 0 explains why).
> Then run the **Setup** cell below. Run cells in order, top to bottom, with **Shift + Enter**.

In [ ]:
# ▶ SETUP - run this cell first. It is safe to run more than once.
REPO_URL = "https://github.com/zshakespear/bd-spectra-onramp.git"
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:                       # running in Google Colab
    ROOT = Path("/content/bd-spectra-onramp")
    if not (ROOT / "bdtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=False)
else:                                                   # running in Jupyter on your own computer
    ROOT = Path.cwd()
    while not (ROOT / "bdtools").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import bdtools as bd
bd.setup_notebook()

## 1. Model grids

A **model grid** is a large set of computed spectra, each for one combination of a few **global parameters**:

| Parameter | Symbol | What it is |
|---|---|---|
| Effective temperature | T<sub>eff</sub> | the temperature of a blackbody with the same total emitted power |
| Surface gravity | log g | log<sub>10</sub> of g in cm s<sup>−2</sup>; young, low-mass objects have low gravity |
| Metallicity | [M/H] | log<sub>10</sub> of the heavy-element fraction relative to the Sun (0 = solar, −0.5 ≈ ⅓ solar) |
| Clouds | f<sub>sed</sub> | sedimentation efficiency: small f<sub>sed</sub> = thick, extended clouds; "nc" = cloud-free |
| Vertical mixing | K<sub>zz</sub> | how fast gas is stirred up from below; strong mixing carries deep-layer chemistry (e.g. CO) upward (disequilibrium chemistry) |

We use a curated subset of model grids: **Sonora Diamondback** (cloudy, equilibrium chemistry; L and T),
**Sonora Flame Skimmer** (cloud-free, disequilibrium chemistry with a K<sub>zz</sub> axis; late T and Y) and **Exo-REM**
(clouds and disequilibrium chemistry). Only Flame Skimmer's 500–550 K models have been downloaded so far, so its
temperature sweep is narrow.
The models were averaged to modest resolution (R ~ 500) once, by your instructor. **Never compare a high-resolution
model directly with SPHEREx data** — bdtools always bins it onto the data's own channels first (notebook 6).

The library is organised around a few **fiducial** models; in each sweep only *one* parameter changes.

In [ ]:
lib = bd.optional(bd.list_models)
lib if lib is None else lib[lib["sweep"] == "fiducial"]

Each plot below has two panels: the spectra (thick line = fiducial) and each model **divided by the fiducial**,
with absorber bands shaded. The ratio panel is where you read the effect of a parameter.

## 2. Effective temperature

**Predict:** a cooler model at the same gravity — will it be fainter everywhere by the same factor?

In [ ]:
r = bd.optional(bd.load_sweep, "L", "teff")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "teff", value_label="Teff [K]", log=True,
                            title="L fiducial: only Teff [K] changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_early", "teff")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "teff", value_label="Teff [K]", log=True,
                            title="T_early fiducial: only Teff [K] changes")

Temperature is a **global** parameter: it changes the overall slope (Planck, notebook 1) *and* the depths of
many bands at once, because the chemistry and the temperature profile both change.

## 3. Surface gravity

In [ ]:
r = bd.optional(bd.load_sweep, "L", "logg")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "logg", value_label="log g", log=True,
                            title="L fiducial: only log g changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_early", "logg")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "logg", value_label="log g", log=True,
                            title="T_early fiducial: only log g changes")

Higher gravity means higher pressure at the photosphere, which strengthens pressure-dependent absorption such as
H<sub>2</sub> CIA (≈1.7–3 µm). Watch the K-band region. The observed signs of **low gravity** (Rustamkulov et al.
2026): a shallower 0.75–1.2 µm slope, weaker water features, and a taller K peak relative to H.

## 4. Metallicity

In [ ]:
r = bd.optional(bd.load_sweep, "L", "mh")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "mh", value_label="[M/H]", log=True,
                            title="L fiducial: only [M/H] changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_early", "mh")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "mh", value_label="[M/H]", log=True,
                            title="T_early fiducial: only [M/H] changes")

Fewer heavy elements → less H<sub>2</sub>O, CH<sub>4</sub>, CO, … but H<sub>2</sub> (and its CIA) is untouched. In observed
**low-metallicity** objects the J (1.3 µm) to K (2.2 µm) peak ratio reverses, because H<sub>2</sub> CIA dominates from
~1.7 to 3 µm (Rustamkulov et al. 2026).

## 5. Clouds

In [ ]:
r = bd.optional(bd.load_sweep, "L", "clouds")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "clouds", value_label="f_sed", log=True,
                            title="L fiducial: only f_sed changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_early", "clouds")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "clouds", value_label="f_sed", log=True,
                            title="T_early fiducial: only f_sed changes")

Thick clouds (small f<sub>sed</sub>) block the deep windows, so the window peaks drop and the spectrum looks redder and
smoother; cloud-free models have the deepest, brightest windows. Clearing clouds is part of why the J band
brightens across the L/T transition.

## 6. Vertical mixing (K<sub>zz</sub>)

In [ ]:
r = bd.optional(bd.load_sweep, "T_late_Y", "kzz")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "kzz", value_label="log Kzz", log=True,
                            title="T_late_Y fiducial: only log Kzz changes")

Stronger mixing dredges up CO (and CO<sub>2</sub>) from hot deep layers faster than chemistry can convert it to CH<sub>4</sub>,
so the 4.3–4.9 µm notches deepen and the 3.3 µm CH<sub>4</sub> band changes. This is **disequilibrium chemistry**; the
Diamondback grid assumes equilibrium and has no K<sub>zz</sub> axis.

The late-T/Y fiducial's other sweeps:

In [ ]:
r = bd.optional(bd.load_sweep, "T_late_Y", "teff")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "teff", value_label="Teff [K]", log=True,
                            title="T_late_Y fiducial: only Teff [K] changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_late_Y", "logg")
if r is not None:
    fid_spec, members = r
    bd.plot_parameter_stack(fid_spec, members, "logg", value_label="log g", log=True,
                            title="T_late_Y fiducial: only log g changes")

## 7. Grids disagree

Even with the **same** parameters, different grids differ by up to about a factor of 2 at some wavelengths
(Rustamkulov et al. 2026). And the best fits to high-S/N data typically still have reduced χ² ≥ 10. **Treat
best-fit parameters as rough.**

"Same parameters" means the same *numbers*, not the same physics: each grid has its own cloud recipe (so
f<sub>sed</sub> = 4 in one grid is not the same cloud as f<sub>sed</sub> = 4 in another), its own chemistry
(Diamondback assumes equilibrium; Exo-REM includes disequilibrium chemistry), and its own opacities. The Exo-REM
spectra in this library also start near 0.93 µm, so the shortest wavelengths show only one grid.

In [ ]:
lib = bd.optional(bd.list_models)
if lib is not None:
    for fid in lib["fiducial"].unique():
        cg = lib[(lib["fiducial"] == fid) & (lib["sweep"] == "cross_grid")]
        f0 = lib[(lib["fiducial"] == fid) & (lib["sweep"] == "fiducial")]
        if cg.empty or f0.empty:
            continue
        a = bd.at_spherex_resolution(bd.load_model(f0.iloc[0]["id"]))
        b = bd.at_spherex_resolution(bd.load_model(cg.iloc[0]["id"]))
        fig, ax = bd.plot_spectrum([a, b], log=True, title=f"{fid}: two grids, same nominal parameters (SPHEREx resolution)")
        print(fid, "largest ratio between the grids:", round(float(np.nanmax(a.flux / b.flux)), 2),
              " smallest:", round(float(np.nanmin(a.flux / b.flux)), 2))

## 8. Real low-gravity and low-metallicity objects

Rustamkulov et al. (2026) include a **low-gravity** sequence at L0 and L5 (their Table 2: field, β = intermediate
gravity, γ = low gravity, i.e. young) and a **low-metallicity** sequence at L0 and L7 (Table 3: field, sd = subdwarf,
esd = extreme subdwarf). Each group below has the same spectral type, so the differences come from gravity or
metallicity rather than temperature. All are normalized at the J peak.

**Predict** before you run it, using sections 3 and 4:
* low gravity: what happens to the 0.75–1.2 µm slope, the water bands, and the K peak relative to H?
* low metallicity: what happens to the K peak (≈2.2 µm) relative to J (≈1.3 µm)?

> Three objects in Tables 2 and 3 are not in the public SPIFF release. The field L0 in both groups is a **stand-in**
> (the Table 1 L0, which the paper itself preferred over the L0 standard for its S/N), and so is the esdL0 (an
> esdL0.5, with low S/N). There is no usdL0 (no ultra-subdwarfs at all in SPIFF). `bd.list_references()` marks
> stand-ins with `status` = `substitute`.

In [ ]:
refs = bd.optional(bd.list_references)
if refs is not None:
    for seqname in ["low_gravity", "low_metallicity"]:
        rows = refs[refs["sequence"] == seqname]
        for base in rows["base_type"].unique():
            group = rows[rows["base_type"] == base].sort_values("order")
            specs = [bd.load_reference(i) for i in group["id"]]
            bd.plot_spectrum(specs, log=True, normalize_at=(1.2, 1.3), errors=False,
                             labels=[s.name.split(" ")[0] for s in specs],
                             title=f"{base}: {seqname.replace('_', ' ')} sequence (normalized at 1.2-1.3 um)")

## 9. Degeneracies: different causes, similar effects

Different parameter changes can look alike in some bands. Here we put a *cooler* model and a *higher-gravity*
model on the same ratio plot.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
FIDUCIAL = "T_early"

In [ ]:
lib = bd.optional(bd.list_models, fiducial=FIDUCIAL)
if lib is not None and not lib.empty:
    fid = bd.load_model(lib[lib["sweep"] == "fiducial"].iloc[0]["id"])
    cool = lib[(lib["sweep"] == "teff") & (lib["teff"] < fid.meta["teff"])].sort_values("teff").tail(1)
    heavy = lib[(lib["sweep"] == "logg") & (lib["logg"] > fid.meta["logg"])].sort_values("logg").head(1)
    if not cool.empty and not heavy.empty:
        fig, ax = plt.subplots()
        bd.shade_bands(ax)
        for row, c in [(cool.iloc[0], bd.LINE_COLORS[1]), (heavy.iloc[0], bd.LINE_COLORS[2])]:
            m = bd.at_spherex_resolution(bd.load_model(row["id"]))
            r = bd.ratio_to(m, bd.at_spherex_resolution(fid))
            ax.plot(m.wave_um, r / np.nanmedian(r), color=c, label=m.name + "  / fiducial (shape only)")
        ax.axhline(1, color="0.4"); ax.legend(fontsize=8); ax.set_xlim(0.7, 5.1)
        ax.set_xlabel("Wavelength [µm]"); ax.set_ylabel("ratio, divided by its median")
        ax.set_title("Cooler vs higher gravity: where do they look alike? where not?")

Dividing by the median removes the overall brightness difference, leaving the shape change. Look for bands where the
two curves move the same way (degenerate) and bands where they differ (these let you tell them apart).

Unresolved **binarity** (two objects blended into one spectrum) can also mimic a parameter change — see notebook 6.

### Exercise 5.1
In the T<sub>eff</sub> sweep, which wavelength range changes most in ratio? Why does that make sense from notebook 1?

<details><summary><b>Answer</b> (try first, then click)</summary>

The short wavelengths (and the deep windows) change most, because they are on the steep, short-wavelength side of
the Planck function, where flux is most sensitive to temperature.

</details>

### Exercise 5.2
Name two things that can make the K peak taller relative to H.

<details><summary><b>Answer</b> (try first, then click)</summary>

Lower gravity (less H<sub>2</sub> CIA; Rustamkulov et al. 2026 list a taller K peak relative to H as a low-gravity sign) and
thicker clouds / redder colours. Metallicity matters too. That's a degeneracy — use other bands to break it.

</details>

## Summary

- T<sub>eff</sub>: global — slope and many bands. log g and [M/H]: act strongly through H<sub>2</sub> CIA (~1.7–3 µm) and band depths.
- Clouds (f<sub>sed</sub>) dim the windows and redden; K<sub>zz</sub> sets how much CO/CO<sub>2</sub> survives in cold atmospheres.
- Grids with the same parameters can differ by ~2×; best fits still have χ²<sub>red</sub> ≥ 10. Parameters are rough.
- Degeneracies are common: look for bands that respond differently.
- Field guide: [Global parameters](../field_guide/parameters/README.md)

**Field guide:** the pages linked above are the short versions to keep open while you work.

## Self-check

Answer these without scrolling up. If one is hard, revisit that section.

1. Which parameter changes the spectrum most globally?
<details><summary><b>Answer</b> (try first, then click)</summary>

Effective temperature.

</details>

2. Name one observed sign of low gravity.
<details><summary><b>Answer</b> (try first, then click)</summary>

A shallower 0.75–1.2 µm slope, weaker water features, or a taller K peak relative to H.

</details>

3. What does vertical mixing do to CO in a cold atmosphere?
<details><summary><b>Answer</b> (try first, then click)</summary>

Brings it up from hot deep layers faster than it converts to CH<sub>4</sub>, so more CO absorption.

</details>

4. Two grids give different spectra for the same parameters. What should you conclude?
<details><summary><b>Answer</b> (try first, then click)</summary>

Model physics differs; treat best-fit parameters as rough.

</details>

**Next:** [Notebook 6 · Fitting models to spectra](06_fitting_models.ipynb)